# Uji Coba TradingAgents: BRIS (Bank Syariah Indonesia)

Notebook uji coba untuk **satu saham IDX**, `BRIS.JK`, dengan konfigurasi hemat (Sonnet di semua tier, debat 1 ronde). Tujuannya memastikan TradingAgents bisa mengambil data saham Indonesia dan menghasilkan laporan, sebelum dipakai lebih luas.

> Hanya untuk riset dan belajar, **bukan saran investasi**.

**Prasyarat:** Python 3.11+, `ANTHROPIC_API_KEY`, akses internet normal (Yahoo Finance dan SEC tidak boleh diblokir).

## 1. Instalasi

In [ ]:
%pip install -q "git+https://github.com/TauricResearch/TradingAgents.git" yfinance plotly pandas

## 2. Pengaturan

Key diminta lewat `getpass`, tidak disimpan di file. Ganti `TICKER` atau `TRADE_DATE` sesuai kebutuhan (`.JK` wajib untuk saham IDX; tanggal pilih hari bursa yang sudah lewat).

In [ ]:
import os, getpass
from tradingagents.default_config import DEFAULT_CONFIG

TICKER = "BRIS.JK"
TRADE_DATE = "2026-10-23"

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

config = DEFAULT_CONFIG.copy()
config["llm_provider"] = "anthropic"
config["deep_think_llm"] = "claude-sonnet-5-5"   # hemat; ganti ke claude-opus-5-5 untuk mutu lebih tinggi
config["quick_think_llm"] = "claude-sonnet-5-5"
config["max_debate_rounds"] = 1
config["max_risk_discuss_rounds"] = 1
config["benchmark_ticker"] = "^JKSE"             # IHSG sebagai pembanding alpha

## 3. Cek data dulu (gratis, tanpa LLM)

Pastikan Yahoo Finance mengenali `BRIS.JK` sebelum menghabiskan biaya API. Kalau kosong, cek koneksi atau kode ticker.

In [ ]:
import yfinance as yf

h = yf.Ticker(TICKER).history(period="3mo")
assert not h.empty, f"Tidak ada data harga untuk {TICKER}. Cek koneksi/ticker sebelum lanjut."
print(f"{len(h)} hari data, terakhir {h.index[-1].date()}, close {h['Close'].iloc[-1]:,.0f} IDR")
h[["Open","High","Low","Close","Volume"]].tail()

## 4. Jalankan analisis

Butuh beberapa menit dan memanggil LLM berkali-kali (ada biaya API).

In [ ]:
from tradingagents.graph.trading_graph import TradingAgentsGraph

ta = TradingAgentsGraph(debug=False, config=config)
state, decision = ta.propagate(TICKER, TRADE_DATE)
report_path = ta.save_reports(state, TICKER)   # markdown + HTML
print("Keputusan:", decision)
print("Laporan:", report_path)

## 5. Baca laporan tiap agent

Bagian yang berisi "withheld" atau "unavailable" berarti sumber datanya tidak terjangkau; itu informasi penting untuk menilai mutu hasil.

In [ ]:
from IPython.display import Markdown, display

for key in ["market_report", "sentiment_report", "news_report", "fundamentals_report", "trader_investment_plan", "final_trade_decision"]:
    display(Markdown(f"### {key}\n\n{state.get(key) or '_(kosong)_'}"))

## 6. Grafik harga + tanggal keputusan

In [ ]:
import pandas as pd
import plotly.graph_objects as go

px = yf.Ticker(TICKER).history(period="1y").reset_index()
fig = go.Figure(go.Scatter(x=pd.to_datetime(px["Date"]).dt.tz_localize(None), y=px["Close"], name=TICKER))
fig.add_vline(x=pd.Timestamp(TRADE_DATE), line_dash="dash", annotation_text="Tanggal keputusan agent")
fig.update_layout(title=f"{TICKER}: harga penutupan 1 tahun", yaxis_title="IDR", height=450)
fig.show()